In [1]:

import os, time, glob, struct
import numpy as np
from cffi import FFI
from pynq import Overlay
from pynq.lib.iic import AxiIIC


os.chdir('/home/xilinx/jupyter_notebooks/footdrop')

# ======================= SETTINGS =======================
ENABLE_ADS = True
ENABLE_FSR = True

# ======================= OVERLAY =======================
ol = Overlay('footdrop.bit')

print("IP blocks:", sorted(ol.ip_dict))

assert 'axi_iic_0' in ol.ip_dict, \
    "axi_iic_0 missing from bitstream"

# ======================= AXI IIC =======================
# Built-in PYNQ driver (replaces manual MMIO)

iic = AxiIIC(ol.ip_dict['axi_iic_0'])
ffi = FFI()

def read_regs(addr, reg, n):
    """Read n consecutive registers via repeated START."""
    if n < 1:
        raise ValueError("n must be >= 1")

    tx = ffi.new("unsigned char[1]")
    rx = ffi.new(f"unsigned char[{n}]")

    tx[0] = reg & 0xFF

    sent = iic.send(addr, tx, 1, option=1)
    received = iic.receive(addr, rx, n, option=0)

    if sent != 1 or received != n:
        raise IOError(
            f"I2C read incomplete at 0x{addr:02X}: "
            f"sent={sent}, received={received}"
        )

    return [int(rx[j]) for j in range(n)]

def write_regs(addr, reg, vals):
    """Write one or more consecutive register bytes."""
    vals = list(vals)

    if not vals:
        raise ValueError("vals cannot be empty")

    tx = ffi.new(f"unsigned char[{len(vals)+1}]")
    tx[0] = reg & 0xFF

    for j, value in enumerate(vals):
        tx[j+1] = value & 0xFF

    sent = iic.send(addr, tx, len(vals)+1, option=0)

    if sent != len(vals)+1:
        raise IOError(
            f"I2C write incomplete at 0x{addr:02X}: "
            f"{sent}/{len(vals)+1} bytes"
        )

def write_reg(addr, reg, val):
    write_regs(addr, reg, [val])

# ======================= BMI160 =======================
BMI160 = 0x68

chip_id = read_regs(BMI160, 0x00, 1)[0]
assert chip_id == 0xD1, \
    f"Incorrect BMI160 chip ID: 0x{chip_id:02X}"

print(f"BMI160 detected at 0x{BMI160:02X}")

# Reset
write_reg(BMI160, 0x7E, 0xB6)
time.sleep(0.1)

# Accelerometer normal mode
write_reg(BMI160, 0x7E, 0x11)
time.sleep(0.05)

# Gyroscope normal mode
write_reg(BMI160, 0x7E, 0x15)
time.sleep(0.1)

# Match original walking-training acquisition
write_reg(BMI160, 0x40, 0x28)  # ACC_CONF: 100 Hz
write_reg(BMI160, 0x41, 0x03)  # ACC_RANGE: +/-2 g
write_reg(BMI160, 0x42, 0x28)  # GYR_CONF: 100 Hz
write_reg(BMI160, 0x43, 0x03)  # GYR_RANGE: +/-2000 dps

ACC_LSB = 16384.0
GYR_LSB = 131.2

def read_imu_frame():
    """
    Returns:
        (ax, ay, az, gx, gy, gz)
    Acceleration in g; angular velocity in dps.
    gy remains index 4 for run_afo.py.
    """
    data = bytes(read_regs(BMI160, 0x0C, 12))
    gx, gy, gz, ax, ay, az = struct.unpack('<6h', data)

    return (
        ax / ACC_LSB,
        ay / ACC_LSB,
        az / ACC_LSB,
        gx / GYR_LSB,
        gy / GYR_LSB,
        gz / GYR_LSB
    )

def verify_bmi160_scale():
    pmu = read_regs(BMI160, 0x03, 1)[0]
    acc_conf = read_regs(BMI160, 0x40, 1)[0]
    acc_range = read_regs(BMI160, 0x41, 1)[0]
    gyr_conf = read_regs(BMI160, 0x42, 1)[0]
    gyr_range = read_regs(BMI160, 0x43, 1)[0]

    print(
        f"BMI160 PMU={pmu:#04x} "
        f"ACC_CONF={acc_conf:#04x} "
        f"ACC_RANGE={acc_range:#04x} "
        f"GYR_CONF={gyr_conf:#04x} "
        f"GYR_RANGE={gyr_range:#04x}"
    )

    assert pmu == 0x14, "BMI160 not in normal mode"
    assert acc_conf == 0x28, "Accelerometer ODR mismatch"
    assert gyr_conf == 0x28, "Gyroscope ODR mismatch"
    assert acc_range == 0x03, "Accelerometer range mismatch"
    assert gyr_range == 0x03, "Gyroscope range mismatch"
    

# ======================= ADS1015 (EMG) =======================
# MyoWare -> ADS1015 -> ISO1540 -> FPGA

EMG_CH = 0
ADS = 0x48

if ENABLE_ADS:
    # Check ADS1015 configuration register
    read_regs(ADS, 0x01, 2)

    # Single-ended AIN0, +/-4.096V,
    # continuous conversion, 3300 SPS
    cfg = (
        ((4 + EMG_CH) << 12)
        | (1 << 9)
        | (0 << 8)
        | (7 << 5)
        | 0x3
    )

    write_regs(
        ADS, 0x01,
        [cfg >> 8, cfg & 0xFF]
    )

    time.sleep(0.01)
    print(f"ADS1015 detected at 0x{ADS:02X}")

def read_emg_raw():
    """
    Returns (raw 12-bit count, volts).
    Index 1 is used by run_afo.py.
    """
    if not ENABLE_ADS:
        raise RuntimeError("ADS1015 disabled")

    d = read_regs(ADS, 0x00, 2)

    raw = ((d[0] << 8) | d[1]) >> 4

    if raw & 0x800:
        raw -= 0x1000

    return raw, raw * 0.002

# ======================= FSRs (XADC) =======================
# A0 = heel, A1 = toe

if ENABLE_FSR:

    if 'xadc_wiz_0' in ol.ip_dict:
        xadc = ol.xadc_wiz_0.mmio

        def read_fsrs():
            """Returns (heel, toe) in volts."""
            return (
                (xadc.read(0x244) >> 4) / 4096 * 3.3,
                (xadc.read(0x264) >> 4) / 4096 * 3.3
            )

    else:
        devices = sorted(
            glob.glob('/sys/bus/iio/devices/iio:device*')
        )
        assert devices, "No XADC IIO device found"

        _dev = devices[0]

        _A0 = glob.glob(
            _dev + '/in_voltage*vaux1_raw'
        )[0]

        _A1 = glob.glob(
            _dev + '/in_voltage*vaux9_raw'
        )[0]

        def _rd(path):
            with open(path) as f:
                return int(f.read())

        def read_fsrs():
            return (
                _rd(_A0) / 4096 * 3.3,
                _rd(_A1) / 4096 * 3.3
            )

else:
    def read_fsrs():
        raise RuntimeError("FSRs disabled")

# ======================= SMOKE TEST =======================
print("\n========== SENSOR CHECK ==========")

verify_bmi160_scale()

imu = read_imu_frame()
print(
    "IMU:",
    tuple(round(v, 3) for v in imu)
)

if ENABLE_ADS:
    emg_raw, emg_volts = read_emg_raw()
    print(
        f"EMG: {emg_volts:.3f} V "
        f"(raw={emg_raw})"
    )

if ENABLE_FSR:
    print(
        "FSR heel/toe:",
        tuple(round(v, 3) for v in read_fsrs())
    )

# ======================= REST OF STACK =======================
%run -n can_mcp.py
%run -n run_afo.py

fpga = FpgaInference(ol)

print("\nFPGA inference initialized")
print("\n========== CELL 0 READY ==========")


IP blocks: ['axi_iic_0', 'axi_quad_spi_0', 'gpio_can', 'gpio_stim', 'inference_stream_0', 'ps7', 'xadc_wiz_0']
BMI160 detected at 0x68
ADS1015 detected at 0x48

========== SENSOR CHECK ==========
BMI160 PMU=0x14 ACC_CONF=0x28 ACC_RANGE=0x03 GYR_CONF=0x28 GYR_RANGE=0x03
IMU: (0.372, -0.062, 0.919, 0.526, -0.282, 0.175)
EMG: 1.546 V (raw=773)
FSR heel/toe: (0.005, 0.004)

FPGA inference initialized

========== CELL 0 READY ==========


In [2]:
_last = {"imu": read_imu_frame(), "emg": read_emg_raw()}
i2c_errors = {"imu": 0, "emg": 0}

def _tolerant(name, fn):
    def wrapped():
        try:
            _last[name] = fn()
        except (IOError, RuntimeError, TimeoutError):
            i2c_errors[name] += 1
        return _last[name]
    return wrapped

read_imu_frame = _tolerant("imu", read_imu_frame)
read_emg_raw   = _tolerant("emg", read_emg_raw)
print("I2C reads are now fault-tolerant")

I2C reads are now fault-tolerant


In [10]:
%run -n can_servo.py
import json, pandas as pd
from safety_layer import SafetyLayer, SafetyConfig
calib = json.load(open("calibration.json"))

# --- CAN + motor ---
spi = AxiSpi(ol.axi_quad_spi_0.mmio)
can = Mcp25625(spi, rst_gpio=(ol.gpio_can.mmio, 0x00, 0))
print("CAN init:", can.init())

FULL = 0.8                                     # full assist, N*m (~1.5x the 0.5 A breakaway)
motor = AKServo(can, motor_id=0x5D, torque_ceiling=FULL)
time.sleep(0.1); print("motor state:", motor.poll_state())

# --- direction + smooth ramp: positive command = toes up ---
class Smooth:
    def __init__(s, m, sign=-1, step_nm=0.1):  # torque can change at most 0.1 N*m per 10 ms
        s.m, s.sign, s.step, s.last = m, sign, step_nm, 0.0
    def command(s, t, **k):
        t = max(s.last - s.step, min(s.last + s.step, t)); s.last = t
        return s.sign * s.m.command(s.sign * t)
    def __getattr__(s, n): return getattr(s.m, n)

drive = Smooth(motor, sign=-1)                 # -1 because negative current lifts the toes
run.__globals__["TORQUE_FULL_NM"] = FULL       # what "full assist" means inside run()

def STOP():
    drive.last = 0.0
    for _ in range(5): motor.current(0.0); time.sleep(0.01)
    print("motor at 0 A")

print("ready -- FULL =", FULL, "N*m")

spi = AxiSpi(ol.axi_quad_spi_0.mmio)
can = Mcp25625(spi, rst_gpio=(ol.gpio_can.mmio, 0x00, 0))
print("CAN init:", can.init())
motor = AKServo(can, motor_id=0x5D, torque_ceiling=0.3)   # hard clamp, N*m
time.sleep(0.1); print("motor state:", motor.poll_state())

run.__globals__["TORQUE_FULL_NM"] = 0.3      # what "full assist" means inside run()

def STOP():
    for _ in range(5): motor.current(0.0); time.sleep(0.01)
    print("motor at 0 A")

CAN init: True
motor state: (-157.0, 0.0, 0.0, 33, 0)
ready -- FULL = 0.8 N*m
CAN init: True
motor state: (-157.0, 0.0, 0.0, 33, 0)


In [4]:
STOP()
input("Hold the footplate at NEUTRAL (90 deg, like standing flat), press Enter")
motor.poll_state(); time.sleep(0.1); POS0 = motor.poll_state()[0]
input("Now hold it at the MOST toes-up it should ever go (~20 deg), press Enter")
motor.poll_state(); time.sleep(0.1); POS_LIM = motor.poll_state()[0]
print(f"neutral {POS0:.1f}   limit {POS_LIM:.1f}   (span {POS_LIM - POS0:+.1f})")

motor at 0 A
Hold the footplate at NEUTRAL (90 deg, like standing flat), press Enter
Now hold it at the MOST toes-up it should ever go (~20 deg), press Enter
neutral -165.5   limit -191.0   (span -25.5)


In [5]:
class Limited:
    def __init__(s, inner, taper=0.3, debug=False):
        s.inner, s.taper, s.debug, s.n = inner, taper, debug, 0
    def command(s, t, **k):
        st = s.inner.poll_state(); t_in, scale, frac = t, 1.0, None
        if t > 0:
            if st is None:
                scale = 0.0
            else:
                frac = (st[0] - POS0) / (POS_LIM - POS0)
                scale = max(0.0, min(1.0, (1.0 - frac) / s.taper))
            t *= scale
        out = s.inner.command(t)
        s.n += 1
        if s.debug and s.n % 10 == 0:
            print(f"req {t_in:.2f}  pos {st[0] if st else None}  frac {frac}  scale {scale:.2f}  "
                  f"ramp {s.inner.last:.2f}  sent {out:+.2f}")
        return out
    def __getattr__(s, n): return getattr(s.inner, n)

drive = Limited(Smooth(motor, sign=-1), debug=True)

def STOP():
    drive.inner.last = 0.0
    for _ in range(5): motor.current(0.0); time.sleep(0.01)
    print("motor at 0 A")

In [6]:
drive.inner.m.ceiling = FULL
print("ceiling:", drive.inner.m.ceiling)      # should print 0.8

ceiling: 0.8


In [11]:
try:
    for name, frac, sec in (("spark", 0.15, 0.6), ("assist", 1.0, 0.5), ("release", 0.0, 0.5)):
        print(name); t_end = time.time() + sec
        while time.time() < t_end:
            drive.command(frac * FULL); drive.poll_state(); time.sleep(0.01)
finally:
    STOP()

spark
assist
release
motor at 0 A


In [8]:
d = pd.read_csv("walk_live_01.csv")
seg = d[(d.t_ms > 20000) & (d.t_ms < 50000)]
L = SafetyLayer(SafetyConfig())
L.calibrate(calib["heel_loaded"], calib["toe_loaded"], calib["heel_unloaded"], calib["toe_unloaded"])

t0, s0, n_assist = time.perf_counter(), seg.t_ms.iloc[0], 0
try:
    for r in seg.itertuples():
        c = L.step(r.t_ms, r.heel, r.toe, r.gy_dps, r.logit, bool(r.logit_valid))
        drive.command(c.torque * FULL); drive.poll_state()
        if c.reason:
            print(f"{(r.t_ms - s0) / 1000:5.1f}s  {c.reason}")
            n_assist += c.reason.startswith("ASSIST")
        while time.perf_counter() < t0 + (r.t_ms - s0) / 1000: pass
finally:
    STOP()
print("assists:", n_assist, "(expect about 13-15)")

  0.0s  STANDING: stance too long
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac None  scale 1.00  ramp 0.00  sent +0.00
req 0.00  pos -166.9  frac N

KeyboardInterrupt: 

In [47]:
try:
    run(fpga, calib, drive, read_emg_raw, read_fsrs, read_imu_frame,
        duration_s=30, log="worn_walk_01.csv")
finally:
    STOP()

pos 527.8   fraction of range 0.07   (0 = neutral, 1 = limit)


current 0.5
    (585.4, 5.061454830783556, 0.539, 35, 0)
motor at 0 A
current 1.0
    (705.1, 9.444724580792172, 1.122, 35, 0)
motor at 0 A
current -1.0
    (609.9, -13.653461405601353, 1.0230000000000001, 35, 0)
motor at 0 A


In [ ]:
from collections import Counter
c = Counter(); t0 = time.perf_counter()
for i in range(1000):
    try: c["ok" if read_regs(BMI160, 0x00, 1)[0] == 0xD1 else "wrong"] += 1
    except Exception as e: c[type(e).__name__] += 1
print(dict(c), f"  {(time.perf_counter() - t0):.2f} ms per read")

In [ ]:
print("EMG read errors so far:", i2c_errors["emg"])

def emg_direct():
    d = read_regs(ADS, 0x00, 2)
    raw = ((d[0] << 8) | d[1]) >> 4
    return (raw - 0x1000 if raw & 0x800 else raw) * 0.002

def grab(label):
    input(f"{label} -- press Enter, hold for 2 s")
    v = np.array([emg_direct() for _ in range(1800)])
    print(f"  {label:8s} mean {v.mean():.3f} V  std {v.std():.4f}  "
          f"min {v.min():.3f}  max {v.max():.3f}  distinct values {len(np.unique(v))}")

grab("relaxed")
grab("flexed")      # pull your toes up as hard as you can

In [ ]:
import glob, pandas as pd
files = sorted(glob.glob("**/*emg*.csv", recursive=True))
print(files[:10])
df = pd.read_csv(files[0])            # pick one walking trial from the list
print(df.columns.tolist())
v = df[df.columns[-1]]                # adjust to the voltage column name printed above
print(f"old trial: mean {v.mean():.3f} V  std {v.std():.4f}  "
      f"1st-99th pct {v.quantile(.01):.3f}-{v.quantile(.99):.3f}")

In [13]:
calib = calibrate(read_emg_raw, read_fsrs, read_imu_frame)

Stand still, relaxed, weight slightly forward. Enter to record 5 s...
Now lift the foot clear of the ground. Enter to record 5 s...
{
  "env_mu": 0.010299353018986573,
  "env_std": 0.008595042892762236,
  "gy_mu": 0.4731606875764659,
  "gy_std": 10.0,
  "gy_std_raw": 2.421877378502021,
  "heel_loaded": 3.1300048828125,
  "toe_loaded": 2.9640380859374997,
  "heel_unloaded": 0.4962890625,
  "toe_unloaded": 0.2135009765625
}


In [16]:
i2c_errors.update(imu=0, emg=0)
run(fpga, calib, motor, read_emg_raw, read_fsrs, read_imu_frame,
    duration_s=150, log="walk_live_03.csv")
print("I2C errors this trial:", i2c_errors)

RUNNING 150 s  (torque ceiling 2.0 N*m, full = 2.0 N*m)
       2 ms  IDLE       
    3000 ms  STANDING   STANDING: stance too long
   19700 ms  IDLE       IDLE: movement resumed
   19710 ms  STANDING   STANDING: stance too long
   19840 ms  IDLE       IDLE: movement resumed
   27310 ms  SPARK      SPARK: intent @ stance 940 ms
   27420 ms  CANCEL     CANCEL: model withdrew
   27620 ms  IDLE       IDLE
   33050 ms  SPARK      SPARK: intent @ stance 1030 ms
   33230 ms  CANCEL     CANCEL: model withdrew
   33440 ms  IDLE       IDLE
   35820 ms  SPARK      SPARK: intent @ stance 1290 ms
   35860 ms  ASSIST     ASSIST: confirmed: toe-off
   36770 ms  REFRACTORY REFRACTORY: veto: assist timeout
   37180 ms  IDLE       IDLE
   38330 ms  SPARK      SPARK: intent @ stance 1060 ms
   38750 ms  CANCEL     CANCEL: model withdrew
   38950 ms  IDLE       IDLE
   43671 ms  SPARK      SPARK: intent @ stance 1120 ms
   44111 ms  CANCEL     CANCEL: model withdrew
   44320 ms  IDLE       IDLE
   48481 m

In [14]:
try:
    run(fpga, calib, drive, read_emg_raw, read_fsrs, read_imu_frame,
        duration_s=30, log="worn_walk_02.csv")
finally:
    STOP()

RUNNING 30 s  (torque ceiling 0.8 N*m, full = 0.3 N*m)
       2 ms  IDLE       
    3000 ms  STANDING   STANDING: stance too long
    3110 ms  IDLE       IDLE: movement resumed
    4620 ms  SPARK      SPARK: intent @ stance 570 ms
    5210 ms  ASSIST     ASSIST: confirmed: gyro +128 dps
    6111 ms  REFRACTORY REFRACTORY: veto: assist timeout
    6520 ms  IDLE       IDLE
    9730 ms  SPARK      SPARK: intent @ stance 810 ms
   10201 ms  CANCEL     CANCEL: model withdrew
   10411 ms  IDLE       IDLE
   10421 ms  SPARK      SPARK: intent @ stance 1501 ms
   10611 ms  ASSIST     ASSIST: confirmed: gyro -128 dps
   11520 ms  REFRACTORY REFRACTORY: veto: assist timeout
   11920 ms  IDLE       IDLE
   12390 ms  SPARK      SPARK: intent @ stance 780 ms
   12901 ms  CANCEL     CANCEL: veto: heel contact
   13113 ms  IDLE       IDLE
   15041 ms  SPARK      SPARK: intent @ stance 690 ms
   15660 ms  CANCEL     CANCEL: veto: heel contact
   15872 ms  IDLE       IDLE
   18450 ms  SPARK      SPARK: